In [11]:

import os
import re
from mem0 import Memory
from typing import Optional
from google import genai #type: ignore
from google.genai import types #type: ignore
from google.colab import drive #type: ignore
from dotenv import load_dotenv


drive.mount("/content/drive")
load_dotenv("/content/drive/MyDrive/.env")
api_key = os.getenv("GOOGLE_API_KEY")


client = genai.Client(api_key=api_key)
MODEL_ID = "gemini-2.5-flash"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
! pip install mem0ai chromadb

In [19]:

# Configuration for mem0
MEM0_CONFIG = {
    "embedder": {
        "provider": "gemini",
        "config": {
            "model": "gemini-embedding-001",
            "embedding_dims": 768,
            "api_key": api_key,
        },
    },
    # Use ChromaDB as a local, in-notebook vector store
    "vector_store": {
        "provider": "chroma",
        "config": {
            "collection_name": "lesson9_memories",
            "path": "/tmp/chroma_mem0",
        },
    },
    "llm": {
        "provider": "gemini",
        "config": {
            "model": MODEL_ID,
            "api_key": api_key,
        },
    },
}

memory = Memory.from_config(MEM0_CONFIG)
MEM_USER_ID = "lesson9_notebook_student"
memory.delete_all(user_id=MEM_USER_ID)
print("✅ Mem0 ready (Gemini embeddings + local Chroma).")

✅ Mem0 ready (Gemini embeddings + local Chroma).


In [20]:
def mem_add_text(text: str, category: str = "semantic", **meta) -> str:
    """Add a single text memory. No LLM is used for extraction or summarization."""
    metadata: dict[str, str | int | float | bool | None]  = {"category": category}
    for k, v in meta.items():
        if isinstance(v, (str, int, float, bool)) or v is None:
            metadata[k] = v
        else:
            metadata[k] = str(v)
    memory.add(text, user_id=MEM_USER_ID, metadata=metadata, infer=False)
    return f"Saved {category} memory."


def mem_search(query: str, limit: int = 5, category: Optional[str] = None) -> list[dict]:
    """
    Category-aware search wrapper.
    Returns the full result dicts so we can inspect metadata.
    """
    res = memory.search(query, user_id=MEM_USER_ID, limit=limit) or {}

    items = res.get("results", [])
    if category is not None:
        items = [r for r in items if (r.get("metadata") or {}).get("category") == category]
    return items

In [21]:
facts = [
    "User prefers vegetarian meals.",
    "User has a dog named George.",
    "User is allergic to gluten.",
    "User's brother is named Mark and is a software engineer.",
]
for f in facts:
    mem_add_text(f, category="semantic")


print(f"Added {len(facts)} semantic memories.")

Added 4 semantic memories.


In [22]:
results = memory.search("brother job", filters={"user_id": MEM_USER_ID}, limit=1)
print(results['results'][0]['memory'])
print(results['results'][0])

User prefers vegetarian meals.
{'id': 'b8281737-c992-460f-8220-37a9da04989f', 'memory': 'User prefers vegetarian meals.', 'hash': '0034073d2dbe31e303972a0599565525', 'metadata': {'category': 'semantic'}, 'score': 0.35457363724708557, 'created_at': '2026-04-20T14:14:27.908265+00:00', 'updated_at': '2026-04-20T14:14:27.908265+00:00', 'user_id': 'lesson9_notebook_student', 'role': 'user'}


In [23]:
dialogue = [
    {"role": "user", "content": "I'm stressed about my project deadline on Friday."},
    {"role": "assistant", "content": "I’m here to help—what’s the blocker?"},
    {"role": "user", "content": "Mainly testing. I also prefer working at night."},
    {"role": "assistant", "content": "Okay, we can split testing into two sessions."},
]

# Ask the LLM to write a clear episodic summary.
episodic_prompt = f"""Summarize the following 3–4 turns as one concise 'episode' (1–2 sentences).
Keep salient details and tone.

{dialogue}
"""
episode_summary = client.models.generate_content(model=MODEL_ID, contents=episodic_prompt)
episode = episode_summary.text.strip()
print(episode)

A user expressed stress over a Friday project deadline, pinpointing testing as the main blocker and mentioning a preference for working at night. The assistant offered support and proposed splitting the testing into two sessions to help manage the workload.


In [24]:
print(
    mem_add_text(
        episode,
        category="episodic",
        summarized=True,
        turns=4, 
    )
)

Saved episodic memory.


In [ ]:
print("\nSearch --> 'deadline stress'\n")
hits = mem_search("deadline stress", limit=1, category="episodic")
for h in hits:
    print(f"{h['memory']}\n")
    print(h)


Search --> 'deadline stress'



ValueError: Top-level entity parameters frozenset({'user_id'}) are not supported in search(). Use filters={'user_id': '...'} instead.